# GraphRAG-Bench — Qwen2.5-3B sur Colab TPU (TorchTPU)

**Objectif** : exécuter les mêmes 24 générations contrôlées (12 questions × TF-IDF/BGE) sur un TPU plutôt que CUDA. Les GPU et TPU utilisent des backends différents ; ce notebook demande TorchTPU et Transformers v5.

**À savoir** : exécution expérimentale ; compatibilité Qwen2.5 avec ton image TPU à qualifier. Commencer par les diagnostics et les 2 réponses du test. Aucune API payante. Les résultats ne sont pas des métriques officielles ; évaluation humaine ultérieure.

Pré-requis : `GraphRAG_Comparaison.zip` téléchargé à la fin du notebook précédent. La session TPU possède son propre système de fichiers temporaire.

## 1. Vérification effective du TPU

In [ ]:
import importlib.util, torch, psutil, shutil
print("RAM totale:", round(psutil.virtual_memory().total / 1024**3, 2), "Go")
print("Disque libre:", round(shutil.disk_usage("/content").free / 1024**3, 2), "Go")
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("torch_tpu installé:", importlib.util.find_spec("torch_tpu") is not None)
if importlib.util.find_spec("torch_tpu") is None:
    raise RuntimeError("TorchTPU n'est pas installé. Voir la cellule installation ci-dessous ou reprendre le runtime T4.")
import torch_tpu
try:
    test = torch.ones(1, device="tpu")
    print("TPU reconnu:", test.device, "| test:", test.cpu().tolist())
except Exception as exc:
    raise RuntimeError("TPU non accessible à PyTorch malgré le runtime Colab") from exc

**Si la cellule 1 échoue uniquement parce que `torch_tpu` est absent**, essayer la cellule facultative suivante, puis redémarrer le runtime via **Exécution → Redémarrer la session**. Ne pas exécuter cette cellule si la première a réussi : un `pip install` superflu peut casser les versions préinstallées de Colab.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("torch_tpu") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch-tpu", "--extra-index-url",
                    "https://download.pytorch.org/whl/cpu"], check=True)
    print("Installation effectuée : redémarrer le runtime avant la cellule 1.")
else:
    print("TorchTPU déjà installé : aucune modification.")

## 2. Choisir un dossier de résultats

Pour sauvegarder les réponses pendant la session, laisser `UTILISER_DRIVE = False`. Pour garder les checkpoints après une coupure de runtime, régler à `True`, ce qui demande explicitement de monter Google Drive.

In [ ]:
from pathlib import Path
UTILISER_DRIVE = False
if UTILISER_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SORTIES = Path("/content/drive/MyDrive/GraphRAG-Bench-LLM-TPU")
else:
    SORTIES = Path("/content/graphrag_llm_tpu_results")
SORTIES.mkdir(parents=True, exist_ok=True)
print("Résultats:", SORTIES)

## 3. Récupérer le code et les dépendances

Le dépôt est cloné depuis la branche TPU, dans un dossier indépendant. `torch_tpu` et `torch` sont conservés, sauf si le diagnostic initial révèle qu'ils manquent.

In [ ]:
import subprocess, sys
REPO = Path("/content/GraphRAG-Benchmark-LLM-TPU")
BRANCHE = "feature/graphrag-pilot-llm-tpu"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCHE,
                    "https://github.com/MamadyIA/GraphRAG-Benchmark.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=5.20,<6", "accelerate>=1.0"], check=True)
import transformers, torch
print("Transformers:", transformers.__version__)
print("TorchTPU opérationnel:", torch.ones(1, device="tpu").cpu().tolist())

## 4. Charger l'archive de récupération partagée

Si la session TPU a supprimé les fichiers précédents, Colab demande d'importer l'archive **GraphRAG_Comparaison.zip** depuis ton ordinateur. L'extraction sélectionne uniquement les deux JSON nécessaires.

In [ ]:
from zipfile import ZipFile
from google.colab import files
INPUTS = Path("/content/graphrag_audit_results")
INPUTS.mkdir(parents=True, exist_ok=True)
NOMS = ("pilot_tfidf_shared_retrieval.json", "pilot_bge_shared_retrieval.json")
if not all((INPUTS / n).is_file() for n in NOMS):
    print("Importe GraphRAG_Comparaison.zip")
    uploads = files.upload()
    archives = [name for name in uploads if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Choisir une seule archive ZIP : GraphRAG_Comparaison.zip")
    with ZipFile(archives[0]) as z:
        for name in NOMS:
            if z.namelist().count(name) != 1:
                raise ValueError("Fichier manquant ou doublonné : " + name)
            (INPUTS / name).write_bytes(z.read(name))

sys.path.insert(0, str(REPO / "experiments"))
from pilot_llm_generate import paired_inputs
pairs = paired_inputs(INPUTS)
assert len(pairs) == 12
print("Prêt :", len(pairs), "questions, 2 méthodes, 24 générations")

## 5. Vérifier les neuf tests et les prompts, sans télécharger Qwen

In [ ]:
SCRIPT = str(REPO / "experiments/pilot_llm_generate.py")
ETAT = SORTIES / "pilot_qwen25_3b_tpu_answers.json"
REVUE = SORTIES / "pilot_qwen25_3b_tpu_review.csv"
BASE = [sys.executable, SCRIPT, "--retrieval-dir", str(INPUTS),
        "--output", str(ETAT), "--review", str(REVUE),
        "--model", "Qwen/Qwen2.5-3B-Instruct",
        "--device", "tpu", "--max-input-tokens", "3584",
        "--max-new-tokens", "384"]
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests",
                "-p", "test_pilot_llm_generate.py", "-v"], cwd=REPO, check=True)
subprocess.run(BASE + ["--dry-run"], cwd=REPO, check=True)
print("Validation hors-modèle réussie.")

## 6. Essai TPU dans le processus du noyau Colab

**Correctif PJRT :** le TPU appartient déjà au processus Python du notebook (pid du noyau Colab). L'ancien `subprocess.run(BASE)` créait un second processus et déclenchait l'erreur « TPU already in use ». Exécuter directement `load_inference` et `generate_paired` ici pour conserver l'unique propriétaire du TPU. Le LLM reste chargé dans la variable `_inference_tpu`. Tout autre problème de compatibilité du backend doit être diagnostiqué indépendamment.

In [ ]:
import sys, json
sys.path.insert(0, str(REPO / "experiments"))
from pilot_llm_generate import paired_inputs, load_inference, generate_paired, export_review

pairs = paired_inputs(INPUTS)
assert len(pairs) == 12

# Le noyau Colab possede deja le TPU. Pas de subprocess pour le LLM.
MODELE = "Qwen/Qwen2.5-3B-Instruct"
if "_inference_tpu" not in globals():
    print("Chargement Qwen dans le processus Colab...")
    _inference_tpu = load_inference(MODELE, "tpu")

state = generate_paired(
    pairs, _inference_tpu, ETAT, MODELE,
    max_new_tokens=384, max_input_tokens=3584,
    limit_questions=1, backend="tpu"
)
export_review(pairs, state, REVUE)
print("Statut:", state["status"], "| reponses:", len(state["completed"]))
for key, item in state["completed"].items():
    print("\n", key, "|", item["generated_tokens"], "tokens")
    print(item["generated_answer"][:1000])

## 7. Génération complète dans le même noyau TPU

Cette cellule réutilise le LLM chargé dans la cellule 6 et reprend les réponses existantes via checkpoint. Ne plus exécuter les anciennes cellules qui lancent un second processus TPU.

In [ ]:
import json

# Reutiliser le modele dans le meme processus PyTorch/TPU.
assert "_inference_tpu" in globals(), "Executer d'abord la cellule de test TPU."
state = generate_paired(
    pairs, _inference_tpu, ETAT, MODELE,
    max_new_tokens=384, max_input_tokens=3584,
    limit_questions=0, backend="tpu"
)
export_review(pairs, state, REVUE)
print("Statut:", state["status"], "| reponses:", len(state["completed"]))
assert state["status"] == "COMPLETE" and len(state["completed"]) == 24
print("24 reponses sauvegardees.")

## 8. Télécharger les résultats

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files
archive = Path("/content/GraphRAG_LLM_Pilote_TPU.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for path in (ETAT, REVUE):
        if path.exists():
            z.write(path, arcname=path.name)
    for name in NOMS:
        z.write(INPUTS / name, arcname=name)
print("Archive :", archive)
files.download(str(archive))

## Précautions

- TPU ≠ GPU CUDA. L'accélération TPU est fournie par **TorchTPU** : il peut y avoir des incompatibilités propres à l'image Colab ou au modèle Qwen2.5, non encore qualifiées sur ton runtime.
- Le code exige `device_map="tpu"` et passe les entrées de génération sur CPU avec `cache_implementation="static"`, selon la documentation récente de Hugging Face.
- La sauvegarde sur le disque `/content` est temporaire. Le montage Drive est facultatif et demande ton autorisation.
- La comparaison est descriptive. Les jugements de correction/fidélité aux sources sont à faire indépendamment, en évitant les fuites de réponses dans les prompts.
- Le modèle de référence utilise la licence **qwen-research** : lire la fiche de licence sur Hugging Face avant diffusion ou usage autre que recherche.
- En cas d'incompatibilité TorchTPU, choisir le GPU T4 (notebook CUDA initial) est la solution de repli la plus fiable ; augmenter la RAM du runtime ne rend pas CUDA disponible.